# **Best Params**


# ---- Prior tuning grids
PRIORS_CONFIG = {
    "gf":      {"L_GRID": [30],  "DECAY_GRID": [0.880]},
    "xgf":     {"L_GRID": [30],  "DECAY_GRID": [0.975]},
    "npxga":   {"L_GRID": [5],  "DECAY_GRID": [0.767]},
    "ppg":     {"L_GRID": [13],  "DECAY_GRID": [0.967]},
}

# ---- XGBoost base params ----
XGB_PARAMS = {
    "objective":        "count:poisson",
    "n_estimators":     2400,
    "learning_rate":    0.014,
    "max_depth":        3,
    "min_child_weight": 4,
    "subsample":        0.96,
    "colsample_bytree": 1.0,
    "random_state":     2026,
    "reg_alpha":        0.0,
    "reg_lambda":       31,
}

# **Config**

In [1]:
# ===============================
# 1) CONFIG
# ===============================

CURRENT_SEASON = "2025-26"

# ---- Time splits ----
TRAIN_END_DATE = "2022-06-30"
VAL_END_DATE   = "2024-06-30"
# Test = everything after VAL_END_DATE

# ---- Goal bucket / clipping (used elsewhere)
GOAL_CLIP = 6

# ---- Prior tuning grids
PRIORS_CONFIG = {
    "gf":      {"L_GRID": [30],  "DECAY_GRID": [0.880]},
    "xgf":     {"L_GRID": [30],  "DECAY_GRID": [0.975]},
    "npxga":   {"L_GRID": [5],  "DECAY_GRID": [0.767]},
    "ppg":     {"L_GRID": [13],  "DECAY_GRID": [0.967]},
}

# ---- XGBoost base params ----
XGB_PARAMS = {
    "objective":        "count:poisson",
    "n_estimators":     2400,
    "learning_rate":    0.014,
    "max_depth":        3,
    "min_child_weight": 4,
    "subsample":        0.96,
    "colsample_bytree": 1.0,
    "random_state":     2026,
    "reg_alpha":        0.0,
    "reg_lambda":       31,
}

# ---- Paths ----
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
INPUTS  = os.path.join(PROJECT_ROOT, "Inputs")
OUTPUTS = os.path.join(PROJECT_ROOT, "Outputs")

# Google Colab users: replace the three lines above with:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = "/content/drive/MyDrive/Football_Prediction_Project"
# INPUTS  = PROJECT_ROOT + "/Inputs"
# OUTPUTS = PROJECT_ROOT + "/Outputs"


# **Data Loading**

In [2]:
# Data root is configured in the CONFIG cell above.
# If using Google Colab, mount your drive there instead.


In [3]:
import numpy as np
import pandas as pd
import os
import glob
import re
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import itertools
import warnings

warnings.filterwarnings("ignore")

In [4]:
import os
import pandas as pd

# --------
# 1) Exact file paths (NO globbing)
# --------
BASE_PATH = os.path.join(INPUTS, "Liga")

PREV_FILE = os.path.join(BASE_PATH, "Previous Seasons", "SP1 Previous Seasons.csv")
CURR_FILE = os.path.join(BASE_PATH, "Current Season",  "SP1 Current Season.csv")

# --------
# 2) Date parsing (yyyy-mm-dd hh:mm:ss)
# --------
def parse_us_date(date_series: pd.Series) -> pd.Series:
    return pd.to_datetime(date_series, errors="coerce")

# --------
# 3) Loader for the new Understat-style files
# --------
def load_prem_two_files(prev_file: str, curr_file: str) -> pd.DataFrame:
    files = [prev_file, curr_file]

    for f in files:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Missing file: {f}")

    print("Files detected:")
    for f in files:
        print("-", f)

    required_cols = [
        "date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_xg",
        "away_xg",
        "home_np_xg",
        "away_np_xg",
        "home_ppda",
        "away_ppda",
        "season",  # yyyy/yyyy e.g. 2025/2026
    ]

    dfs = []
    for f in files:
        base = os.path.basename(f)

        df = pd.read_csv(f)
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{base} is missing columns: {missing}")

        df = df[required_cols].copy()

        # Parse date
        df["date"] = parse_us_date(df["date"])
        df = df.dropna(subset=["date", "home_team", "away_team", "season", "home_goals", "away_goals"])

        # Numerics
        num_cols = [
            "home_goals", "away_goals",
            "home_xg", "away_xg",
            "home_np_xg", "away_np_xg",
            "home_ppda", "away_ppda",
        ]
        for c in num_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        # Drop unplayed rows (missing goals)
        df = df.dropna(subset=["home_goals", "away_goals"])

        # Goals as ints
        df["home_goals"] = df["home_goals"].astype(int)
        df["away_goals"] = df["away_goals"].astype(int)

        # Season string as provided
        df["season"] = df["season"].astype(str)

        df["source_file"] = base
        dfs.append(df)

    full = pd.concat(dfs, ignore_index=True).sort_values("date").reset_index(drop=True)

    # Rename to match your existing pipeline's column names
    full = full.rename(columns={
        "date": "Date",
        "home_team": "HomeTeam",
        "away_team": "AwayTeam",
        "home_goals": "FTHG",
        "away_goals": "FTAG",
        # keep these as-is (already named well)
        "home_xg": "home_xg",
        "away_xg": "away_xg",
        "home_np_xg": "home_np_xg",
        "away_np_xg": "away_np_xg",
        "home_ppda": "home_ppda",
        "away_ppda": "away_ppda",
    })

    # Summary
    print("\nLoaded seasons:", sorted(full["season"].unique()))
    print("Total matches:", len(full))
    print("Date range:", full["Date"].min(), "→", full["Date"].max())
    print("\nMatches per season:\n", full.groupby("season").size().sort_index())

    return full

# --------
# 4) Run
# --------
matches = load_prem_two_files(PREV_FILE, CURR_FILE)

# matches columns now include:
# Date, HomeTeam, AwayTeam, FTHG, FTAG, season, source_file
# plus: home_xg, away_xg, home_np_xg, away_np_xg, home_ppda, away_ppda

Files detected:
- /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Liga/Previous Seasons/SP1 Previous Seasons.csv
- /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Liga/Current Season/SP1 Current Season.csv

Loaded seasons: ['2014/2015', '2015/2016', '2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Total matches: 4540
Date range: 2014-08-23 18:00:00 → 2026-05-14 19:30:00

Matches per season:
 season
2014/2015    380
2015/2016    380
2016/2017    380
2017/2018    380
2018/2019    380
2019/2020    380
2020/2021    380
2021/2022    380
2022/2023    380
2023/2024    380
2024/2025    380
2025/2026    360
dtype: int64


In [5]:
matches

,Date,HomeTeam,AwayTeam,FTHG,FTAG,home_xg,away_xg,home_np_xg,away_np_xg,home_ppda,away_ppda,season,source_file
0,2014-08-23 18:00:00,Malaga,Athletic Club,1,0,1.321070,1.141510,0.438073,1.141510,12.071429,6.300000,2014/2015,SP1 Previous Seasons.csv
1,2014-08-23 20:00:00,Granada,Deportivo La Coruna,2,1,0.548351,0.379371,0.548351,0.379371,4.775000,5.800000,2014/2015,SP1 Previous Seasons.csv
2,2014-08-23 20:00:00,Sevilla,Valencia,1,1,1.171970,1.749030,1.171970,1.749030,9.416667,10.650000,2014/2015,SP1 Previous Seasons.csv
3,2014-08-23 22:00:00,Almeria,Espanyol,1,1,0.978788,0.399306,0.978788,0.399306,6.666667,15.800000,2014/2015,SP1 Previous Seasons.csv
4,2014-08-24 18:00:00,Eibar,Real Sociedad,1,0,0.465398,0.975523,0.465398,0.975523,8.791667,6.666667,2014/2015,SP1 Previous Seasons.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4535,2026-05-13 19:30:00,Alaves,Barcelona,1,0,0.953470,0.356356,0.953470,0.356356,17.347826,3.625000,2025/2026,SP1 Current Season.csv
4536,2026-05-13 19:30:00,Getafe,Mallorca,3,1,1.821080,0.412059,1.821080,0.412059,11.961538,8.941176,2025/2026,SP1 Current Season.csv
4537,2026-05-14 17:00:00,Valencia,Rayo Vallecano,1,1,0.938336,1.530410,0.938336,0.787128,10.045455,6.378378,2025/2026,SP1 Current Season.csv
4538,2026-05-14 18:00:00,Girona,Real Sociedad,1,1,2.685860,0.553216,2.685860,0.553216,9.884615,9.900000,2025/2026,SP1 Current Season.csv


# **Build Tables**

In [6]:
# ===============================
# 3) BUILD FIXTURES + TEAM_MATCHES
# ===============================

matches = matches.sort_values("Date").reset_index(drop=True)
matches["fixture_id"] = matches.index

# ---- Fixtures: one row per match ----
fixtures = matches[[
    "fixture_id",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "home_xg",
    "away_xg",
    "home_np_xg",
    "away_np_xg",
    "season"
]].copy()

fixtures.columns = [
    "fixture_id",
    "match_date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_np_xg",
    "away_np_xg",
    "season"
]

# ---- Team perspective rows (two rows per fixture) ----
home_rows = fixtures.copy()
home_rows["team"]          = home_rows["home_team"]
home_rows["opponent"]      = home_rows["away_team"]
home_rows["is_home"]       = 1
home_rows["goals_for"]     = home_rows["home_goals"]
home_rows["goals_against"] = home_rows["away_goals"]
home_rows["xg_for"]        = home_rows["home_xg"]
home_rows["np_xg_for"]     = home_rows["home_np_xg"]
home_rows["np_xg_against"] = home_rows["away_np_xg"]

away_rows = fixtures.copy()
away_rows["team"]          = away_rows["away_team"]
away_rows["opponent"]      = away_rows["home_team"]
away_rows["is_home"]       = 0
away_rows["goals_for"]     = away_rows["away_goals"]
away_rows["goals_against"] = away_rows["home_goals"]
away_rows["xg_for"]        = away_rows["away_xg"]
away_rows["np_xg_for"]     = away_rows["away_np_xg"]
away_rows["np_xg_against"] = away_rows["home_np_xg"]

# ---- Points (3/1/0) ----
def calc_points(row):
    if row["goals_for"] > row["goals_against"]:   return 3
    elif row["goals_for"] == row["goals_against"]: return 1
    else:                                          return 0

team_matches = pd.concat([home_rows, away_rows], ignore_index=True)
team_matches["points"] = team_matches.apply(calc_points, axis=1)

team_matches = team_matches[[
    "fixture_id",
    "match_date",
    "season",
    "team",
    "opponent",
    "is_home",
    "goals_for",
    "xg_for",
    "np_xg_against",
    "points",
]].sort_values(["team", "match_date"]).reset_index(drop=True)

print("Team perspective rows:", len(team_matches))

Team perspective rows: 9080


# **Rest Days**

In [7]:
# ===============================
# REST DAYS FEATURE
# ===============================

import os

HIST_FIXTURES_PATH    = os.path.join(INPUTS, "Fixtures", "historic_competitions_fixtures.csv")
CURRENT_FIXTURES_PATH = os.path.join(INPUTS, "Fixtures", f"all_competitions_fixtures_{CURRENT_SEASON}.csv")

# ---- Load and concat all-comp fixtures ----
dfs = []
for path in [HIST_FIXTURES_PATH, CURRENT_FIXTURES_PATH]:
    if os.path.exists(path):
        dfs.append(pd.read_csv(path, parse_dates=["date"]))

if dfs:
    all_comp = pd.concat(dfs, ignore_index=True)
    all_comp["date"] = pd.to_datetime(all_comp["date"]).dt.normalize()
    all_comp = all_comp.drop_duplicates(["date", "team"]).sort_values(["team", "date"]).reset_index(drop=True)
    all_comp_teams = set(all_comp["team"].unique())
    print(f"All-comp fixtures loaded: {len(all_comp)} rows, {len(all_comp_teams)} teams")
else:
    all_comp = pd.DataFrame(columns=["date", "team"])
    all_comp_teams = set()
    print("No all-comp fixture files found — rest days will use league fixtures only")

# ---- Unique teams in league data ----
league_teams = set(team_matches["team"].unique())
print(f"League teams: {len(league_teams)}")
print(f"League teams missing from all-comp: {sorted(league_teams - all_comp_teams)}")

# ---- Normalise match_date to date only for comparison ----
team_matches["match_date"] = pd.to_datetime(team_matches["match_date"]).dt.normalize()

# ---- Build per-team fixture timeline ----
# For each league team: use all-comp if available, else fall back to league dates only
league_dates_df = (
    team_matches[["team", "match_date"]]
    .drop_duplicates()
    .rename(columns={"match_date": "date"})
    .copy()
)

def build_team_timeline(team_name):
    if team_name in all_comp_teams:
        dates = all_comp[all_comp["team"] == team_name][["date"]].copy()
        dates["is_league"] = 0  # will mark league games below
    else:
        # Fallback — league only
        dates = league_dates_df[league_dates_df["team"] == team_name][["date"]].copy()
        dates["is_league"] = 1

    # Mark which rows are league fixtures
    league_dates_for_team = set(
        league_dates_df[league_dates_df["team"] == team_name]["date"]
    )
    dates["is_league"] = dates["date"].isin(league_dates_for_team).astype(int)

    dates = dates.drop_duplicates("date").sort_values("date").reset_index(drop=True)
    dates["rest_days"] = dates["date"].diff().dt.days  # days since previous game (any comp)
    return dates

# ---- Compute rest days for every team_matches row ----
rest_days_map = {}  # (team, date) -> rest_days

for team in league_teams:
    timeline = build_team_timeline(team)
    for _, row in timeline[timeline["is_league"] == 1].iterrows():
        rest_days_map[(team, row["date"])] = row["rest_days"]

# ---- Join back to team_matches ----
team_matches["rest_days"] = team_matches.apply(
    lambda r: rest_days_map.get((r["team"], r["match_date"]), np.nan), axis=1
)

print(f"\nRest days computed.")
print(f"NaN count: {team_matches['rest_days'].isna().sum()} (first game per team, expected)")
print(team_matches["rest_days"].describe())

All-comp fixtures loaded: 79452 rows, 152 teams
League teams: 32
League teams missing from all-comp: []

Rest days computed.
NaN count: 162 (first game per team, expected)
count    8918.000000
mean        7.278986
std         8.654910
min         2.000000
25%         4.000000
50%         6.000000
75%         8.000000
max        99.000000
Name: rest_days, dtype: float64


# **Promoted**

In [8]:
# ===============================
# PROMOTED TEAMS FEATURE
# ===============================

# Teams promoted into the top flight at the START of our data (2014/15)
# i.e. were not in the league in 2013/14
BOOTSTRAP_PROMOTED = {"Eibar", "Deportivo La Coruna", "Cordoba"}

def compute_promoted(team_matches_df, bootstrap_promoted):
    df = team_matches_df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])
    df["is_promoted"] = 0

    # Get all unique (team, season) combos
    team_seasons = df[["team", "season"]].drop_duplicates()

    # Get sorted unique seasons
    seasons = sorted(df["season"].unique())

    promoted_by_season = {}

    for i, season in enumerate(seasons):
        if i == 0:
            # First season — mark bootstrap promoted teams
            promoted = bootstrap_promoted
        else:
            prev_season = seasons[i - 1]
            teams_this  = set(team_seasons[team_seasons["season"] == season]["team"])
            teams_prev  = set(team_seasons[team_seasons["season"] == prev_season]["team"])
            promoted     = teams_this - teams_prev

        if promoted:
            mask = (df["season"] == season) & (df["team"].isin(promoted))
            df.loc[mask, "is_promoted"] = 1
            print(f"  {season}: promoted = {sorted(promoted)}")

        promoted_by_season[season] = promoted

    return df, promoted_by_season

team_matches, promoted_by_season = compute_promoted(team_matches, BOOTSTRAP_PROMOTED)

# Save current season's promoted teams for use in output block
current_season = team_matches["season"].max()
promoted_current_season = promoted_by_season[current_season]
print(f"\nCurrent season ({current_season}) promoted teams: {sorted(promoted_current_season)}")

print(f"\nis_promoted distribution:\n{team_matches['is_promoted'].value_counts()}")

  2014/2015: promoted = ['Cordoba', 'Deportivo La Coruna', 'Eibar']
  2015/2016: promoted = ['Las Palmas', 'Real Betis', 'Sporting Gijon']
  2016/2017: promoted = ['Alaves', 'Leganes', 'Osasuna']
  2017/2018: promoted = ['Getafe', 'Girona', 'Levante']
  2018/2019: promoted = ['Rayo Vallecano', 'Real Valladolid', 'SD Huesca']
  2019/2020: promoted = ['Granada', 'Mallorca', 'Osasuna']
  2020/2021: promoted = ['Cadiz', 'Elche', 'SD Huesca']
  2021/2022: promoted = ['Espanyol', 'Mallorca', 'Rayo Vallecano']
  2022/2023: promoted = ['Almeria', 'Girona', 'Real Valladolid']
  2023/2024: promoted = ['Alaves', 'Granada', 'Las Palmas']
  2024/2025: promoted = ['Espanyol', 'Leganes', 'Real Valladolid']
  2025/2026: promoted = ['Elche', 'Levante', 'Real Oviedo']

Current season (2025/2026) promoted teams: ['Elche', 'Levante', 'Real Oviedo']

is_promoted distribution:
is_promoted
0    7718
1    1362
Name: count, dtype: int64


# **Strength Function**

In [9]:
# ===============================
# 4) DECAYED MEAN (NO MAX_LOOKBACK CAP)
# ===============================

def decayed_mean_last_L(values, L, decay):
    """
    Weighted mean of the last L values (or all values if fewer than L).
    Most recent match gets weight 1, then decay, decay^2, ...

    Convention:
      - If L == 0: treat as "feature disabled" and return 0.0 (constant).
    """
    if int(L) == 0:
        return 0.0

    if len(values) == 0:
        return np.nan

    vals = np.asarray(values, dtype=float)
    vals = vals[-min(L, len(vals)):]  # only cap is L
    n = len(vals)

    weights = np.asarray([decay**k for k in range(n)][::-1], dtype=float)
    return float(np.dot(vals, weights) / weights.sum())


# ===============================
# 4B) 8 INDEPENDENT PRIOR FUNCTIONS
# ===============================
# Each function is just a thin wrapper so you can tune each with its own (L, decay).

def prior_goals_for(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_goals_against(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_xg_for(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_xg_against(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_npxg_for(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_npxg_against(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_ppda_for(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

def prior_ppda_against(values, L, decay):
    return decayed_mean_last_L(values, L=L, decay=decay)

# Venue Specific PPG

def apply_venue_ppg_team_only(model_df):
    df = model_df.copy()
    has_home = "team_ppg_home_prior" in df.columns
    has_away = "team_ppg_away_prior" in df.columns
    if has_home and has_away:
        df["team_ppg_venue"] = np.where(
            df["is_home"] == 1,
            df["team_ppg_home_prior"],
            df["team_ppg_away_prior"]
        )
    df = df.drop(columns=[
        "team_ppg_home_prior", "team_ppg_away_prior",
        "opp_ppg_home_prior",  "opp_ppg_away_prior",
    ], errors="ignore")
    return df

# **Priors**

In [10]:
# ===============================
# 5) COMPUTE PRIORS
# ===============================

def compute_priors(team_matches: pd.DataFrame, priors_params: dict) -> pd.DataFrame:
    """
    Computes pre-match priors per team, using independent (L, decay) per feature.

    Expects team_matches to contain:
      goals_for, xg_for, np_xg_against, points, is_home

    priors_params example:
    {
      "gf":    {"L": 24,  "decay": 0.89895},
      "xgf":   {"L": 26,  "decay": 0.994},
      "npxga": {"L": 38,  "decay": 0.99927},
      "ppg":   {"L": 20,  "decay": 0.95},   # shared for ppg_home and ppg_away
    }
    """

    df = team_matches.copy()

    out_cols = [
        "gf_prior",
        "xgf_prior",
        "npxga_prior",
        "ppg_home_prior",
        "ppg_away_prior",
    ]
    for c in out_cols:
        df[c] = np.nan

    for team in df["team"].unique():
        sub = df[df["team"] == team].sort_values("match_date")

        gf_hist       = []
        xgf_hist      = []
        npxga_hist    = []
        ppg_home_hist = []
        ppg_away_hist = []

        for i in sub.index:
            if gf_hist:
                df.loc[i, "gf_prior"] = decayed_mean_last_L(
                    gf_hist, priors_params["gf"]["L"], priors_params["gf"]["decay"]
                )
                df.loc[i, "xgf_prior"] = decayed_mean_last_L(
                    xgf_hist, priors_params["xgf"]["L"], priors_params["xgf"]["decay"]
                )
                df.loc[i, "npxga_prior"] = decayed_mean_last_L(
                    npxga_hist, priors_params["npxga"]["L"], priors_params["npxga"]["decay"]
                )

            # ppg_home and ppg_away only compute once we have at least 1 game in each bucket
            if ppg_home_hist:
                df.loc[i, "ppg_home_prior"] = decayed_mean_last_L(
                    ppg_home_hist, priors_params["ppg"]["L"], priors_params["ppg"]["decay"]
                )
            if ppg_away_hist:
                df.loc[i, "ppg_away_prior"] = decayed_mean_last_L(
                    ppg_away_hist, priors_params["ppg"]["L"], priors_params["ppg"]["decay"]
                )

            # Update histories — no leakage
            gf_hist.append(df.loc[i, "goals_for"])
            xgf_hist.append(df.loc[i, "xg_for"])
            npxga_hist.append(df.loc[i, "np_xg_against"])

            # Points split by home/away
            if df.loc[i, "is_home"] == 1:
                ppg_home_hist.append(df.loc[i, "points"])
            else:
                ppg_away_hist.append(df.loc[i, "points"])

    # Drop points now — no longer needed, prevent leakage
    df = df.drop(columns=["points"])

    return df

# **Modelling Table**

In [11]:
# ===============================
# 6) BUILD MODEL TABLE
# ===============================

KEY_TO_COLS = {
    "gf":    ["team_att_prior",     "opp_att_prior"],
    "xgf":   ["team_xgf_prior"],
    "npxga": ["team_npxga_prior",   "opp_npxga_prior"],
    "ppg":   ["team_ppg_home_prior", "opp_ppg_home_prior",
              "team_ppg_away_prior", "opp_ppg_away_prior"],
}

def build_model_table(df: pd.DataFrame, priors_params: dict) -> pd.DataFrame:

    priors_cols = [
        "gf_prior",
        "xgf_prior",
        "npxga_prior",
        "ppg_home_prior",
        "ppg_away_prior",
    ]

    # Merge opponent priors
    merged = df.merge(
        df[["fixture_id", "team", "rest_days", "is_promoted"] + priors_cols],
        left_on=["fixture_id", "opponent"],
        right_on=["fixture_id", "team"],
        suffixes=("", "_opp")
    )

    model_df = merged[[
        "match_date",
        "fixture_id",
        "team",
        "opponent",
        "is_home",
        "rest_days_opp",
        "is_promoted_opp",

        # Team priors
        "gf_prior",
        "xgf_prior",
        "npxga_prior",
        "ppg_home_prior",
        "ppg_away_prior",

        # Opponent priors
        "gf_prior_opp",
        "npxga_prior_opp",
        "ppg_home_prior_opp",
        "ppg_away_prior_opp",

        # Target
        "goals_for"
    ]].copy()

    model_df = model_df.rename(columns={
        "goals_for":          "goals_scored",
        "rest_days_opp":      "opp_rest_days",
        "is_promoted_opp":    "opp_is_promoted",

        "gf_prior":           "team_att_prior",
        "xgf_prior":          "team_xgf_prior",
        "npxga_prior":        "team_npxga_prior",
        "ppg_home_prior":     "team_ppg_home_prior",
        "ppg_away_prior":     "team_ppg_away_prior",

        "gf_prior_opp":       "opp_att_prior",
        "npxga_prior_opp":    "opp_npxga_prior",
        "ppg_home_prior_opp": "opp_ppg_home_prior",
        "ppg_away_prior_opp": "opp_ppg_away_prior",
    })

    # Drop columns for any disabled (L=0) priors
    cols_to_drop = []
    for k, cfg in priors_params.items():
        if int(cfg["L"]) == 0:
            cols_to_drop.extend(KEY_TO_COLS.get(k, []))
    model_df = model_df.drop(columns=[c for c in cols_to_drop if c in model_df.columns])

    # Create team_ppg_venue and drop raw ppg columns
    model_df = apply_venue_ppg_team_only(model_df)

    return model_df.dropna().reset_index(drop=True)

# **Log Loss**

In [12]:
# ===============================
# 7) LOG-LOSS SCORING (POISSON DISTRIBUTION)
# ===============================

import numpy as np
import math

GOAL_MAX = 6  # bucket 6+

def poisson_probs(lam, max_goal=GOAL_MAX):
    """
    Returns array of probabilities:
    [P(0), P(1), ..., P(max_goal-1), P(max_goal+)]
    """
    lam = max(float(lam), 1e-10)  # numerical safety

    probs = []
    for k in range(max_goal):
        pk = math.exp(-lam) * lam**k / math.factorial(k)
        probs.append(pk)

    tail = 1.0 - sum(probs)
    probs.append(max(tail, 0.0))  # P(max_goal+)

    return np.array(probs)


def log_loss_poisson(y_true, pred_lambda, max_goal=GOAL_MAX):
    """
    y_true: array of actual goals (integers)
    pred_lambda: array of model outputs (floats)

    Returns mean log loss (lower is better)
    """
    losses = []

    for yt, lam in zip(y_true, pred_lambda):
        probs = poisson_probs(lam, max_goal=max_goal)

        if yt >= max_goal:
            p = probs[-1]
        else:
            p = probs[int(yt)]

        # avoid log(0)
        p = max(p, 1e-15)

        losses.append(-np.log(p))

    return float(np.mean(losses))

# **Time Split**

In [13]:
# ===============================
# 8) TIME SPLIT
# ===============================

def time_split(df):
    df = df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])

    train = df[df["match_date"] <= TRAIN_END_DATE]
    val = df[(df["match_date"] > TRAIN_END_DATE) &
             (df["match_date"] <= VAL_END_DATE)]
    test = df[df["match_date"] > VAL_END_DATE]

    return train, val, test

# **Grid Search**

In [14]:
# ===============================
# 9) GRID SEARCH (LOG LOSS)
# ===============================

import itertools
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

NON_FEATURE_COLS = {"match_date", "fixture_id", "team", "opponent", "goals_scored"}

grid = itertools.product(
    PRIORS_CONFIG["gf"]["L_GRID"],    PRIORS_CONFIG["gf"]["DECAY_GRID"],
    PRIORS_CONFIG["xgf"]["L_GRID"],   PRIORS_CONFIG["xgf"]["DECAY_GRID"],
    PRIORS_CONFIG["npxga"]["L_GRID"], PRIORS_CONFIG["npxga"]["DECAY_GRID"],
    PRIORS_CONFIG["ppg"]["L_GRID"],   PRIORS_CONFIG["ppg"]["DECAY_GRID"],
)

results = []
i = 0

for (
    L_gf,    d_gf,
    L_xgf,   d_xgf,
    L_npxga, d_npxga,
    L_ppg,   d_ppg,
) in grid:
    i += 1
    print(
        f"Test {i}: "
        f"gf({L_gf},{d_gf}) "
        f"xgf({L_xgf},{d_xgf}) "
        f"npxga({L_npxga},{d_npxga}) "
        f"ppg({L_ppg},{d_ppg})"
    )

    priors_params = {
        "gf":    {"L": int(L_gf),    "decay": float(d_gf)},
        "xgf":   {"L": int(L_xgf),   "decay": float(d_xgf)},
        "npxga": {"L": int(L_npxga), "decay": float(d_npxga)},
        "ppg":   {"L": int(L_ppg),   "decay": float(d_ppg)},
    }

    priors_df = compute_priors(team_matches, priors_params=priors_params)
    model_df  = build_model_table(priors_df, priors_params)

    train, val, test = time_split(model_df)
    if len(train) == 0 or len(val) == 0:
        continue

    feature_cols = [c for c in model_df.columns if c not in NON_FEATURE_COLS]

    X_train = train[feature_cols];  y_train = train["goals_scored"].astype(int)
    X_val   = val[feature_cols];    y_val   = val["goals_scored"].astype(int)

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train)

    score = log_loss_poisson(y_val.values, model.predict(X_val))

    results.append({
        "L_gf":    L_gf,    "decay_gf":    d_gf,
        "L_xgf":   L_xgf,   "decay_xgf":   d_xgf,
        "L_npxga": L_npxga, "decay_npxga": d_npxga,
        "L_ppg":   L_ppg,   "decay_ppg":   d_ppg,
        "val_logloss":  score,
        "feature_cols": feature_cols,
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("val_logloss", ascending=True)
    .reset_index(drop=True)
)

results_df.head()

Test 1: gf(30,0.88) xgf(30,0.975) npxga(5,0.767) ppg(13,0.967)


,L_gf,decay_gf,L_xgf,decay_xgf,L_npxga,decay_npxga,L_ppg,decay_ppg,val_logloss,feature_cols
0,30,0.88,30,0.975,5,0.767,13,0.967,1.413671,"[is_home, opp_rest_days, opp_is_promoted, team..."


In [15]:
results_df.head(25)

,L_gf,decay_gf,L_xgf,decay_xgf,L_npxga,decay_npxga,L_ppg,decay_ppg,val_logloss,feature_cols
0,30,0.88,30,0.975,5,0.767,13,0.967,1.413671,"[is_home, opp_rest_days, opp_is_promoted, team..."


# **Final Model**

In [16]:
# ===============================
# FINAL MODEL (pick absolute best from grid search) + TEST EVALUATION
# ===============================

best_row  = results_df.sort_values("val_logloss", ascending=True).iloc[0]
best_loss = float(best_row["val_logloss"])

best_priors_params = {
    "gf":    {"L": int(best_row["L_gf"]),    "decay": float(best_row["decay_gf"])},
    "xgf":   {"L": int(best_row["L_xgf"]),   "decay": float(best_row["decay_xgf"])},
    "npxga": {"L": int(best_row["L_npxga"]), "decay": float(best_row["decay_npxga"])},
    "ppg":   {"L": int(best_row["L_ppg"]),   "decay": float(best_row["decay_ppg"])},
}

FEATURE_COLS = best_row["feature_cols"]

print("Best VAL Log Loss:", best_loss)
print("Best priors_params:", best_priors_params)
print("Feature cols:", FEATURE_COLS)

priors_df = compute_priors(team_matches, priors_params=best_priors_params)
model_df  = build_model_table(priors_df, best_priors_params)

train, val, test = time_split(model_df)

train_full   = pd.concat([train, val], ignore_index=True)
X_train_full = train_full[FEATURE_COLS]
y_train_full = train_full["goals_scored"].astype(int)

X_test = test[FEATURE_COLS]
y_test = test["goals_scored"].astype(int)

final_model = XGBRegressor(**XGB_PARAMS)
final_model.fit(X_train_full, y_train_full)

test_preds_float = final_model.predict(X_test)

test_logloss = log_loss_poisson(y_test.values, test_preds_float)
print("Final Test Log Loss:", test_logloss)

Best VAL Log Loss: 1.413671125180296
Best priors_params: {'gf': {'L': 30, 'decay': 0.88}, 'xgf': {'L': 30, 'decay': 0.975}, 'npxga': {'L': 5, 'decay': 0.767}, 'ppg': {'L': 13, 'decay': 0.967}}
Feature cols: ['is_home', 'opp_rest_days', 'opp_is_promoted', 'team_att_prior', 'team_xgf_prior', 'team_npxga_prior', 'opp_att_prior', 'opp_npxga_prior', 'team_ppg_venue']
Final Test Log Loss: 1.3903111162312876


# **Diagnostics**

In [17]:
# ===============================
# 11) DIAGNOSTICS (LOG LOSS / PROBABILISTIC + OVER/UNDER READ)
# ===============================

# 1) Primary metric
print("Test Log Loss:", log_loss_poisson(y_test.values, test_preds_float))

# 2) Sanity metric (lambda vs true goals)
print("MAE (lambda vs true):", float(np.mean(np.abs(test_preds_float - y_test.values))))

# 3) Most likely goal bucket (0..6+)
def most_likely_bucket(pred_lambda, max_goal=GOAL_CLIP):
    probs = poisson_probs(pred_lambda, max_goal=max_goal)
    return int(np.argmax(probs))

y_true_bucket = np.minimum(y_test.values.astype(int), GOAL_CLIP)
y_pred_bucket = np.array(
    [most_likely_bucket(lam, max_goal=GOAL_CLIP) for lam in test_preds_float],
    dtype=int
)

err_bucket = y_pred_bucket - y_true_bucket

print("Top-1 bucket accuracy (0..6+):", float(np.mean(err_bucket == 0)))
print("Within ±1 bucket rate:", float(np.mean(np.abs(err_bucket) <= 1)))
print("Over by 1 bucket rate:", float(np.mean(err_bucket == 1)))
print("Under by 1 bucket rate:", float(np.mean(err_bucket == -1)))

# 4) Over / Under tendency based on lambda vs actual
y_true   = y_test.values.astype(float)
y_lambda = test_preds_float

over_rate         = np.mean(y_lambda > y_true)
under_rate        = np.mean(y_lambda < y_true)
exact_lambda_rate = np.mean(np.isclose(y_lambda, y_true))

print("Lambda > Actual (Overestimation rate):", float(over_rate))
print("Lambda < Actual (Underestimation rate):", float(under_rate))
print("Lambda == Actual (Exact float match rate):", float(exact_lambda_rate))

Test Log Loss: 1.3903111162312876
MAE (lambda vs true): 0.8406619510004076
Top-1 bucket accuracy (0..6+): 0.38305084745762713
Within ±1 bucket rate: 0.8413559322033898
Over by 1 bucket rate: 0.2
Under by 1 bucket rate: 0.2583050847457627
Lambda > Actual (Overestimation rate): 0.5972881355932204
Lambda < Actual (Underestimation rate): 0.40271186440677964
Lambda == Actual (Exact float match rate): 0.0


In [18]:
model_df

,match_date,fixture_id,team,opponent,is_home,opp_rest_days,opp_is_promoted,team_att_prior,team_xgf_prior,team_npxga_prior,opp_att_prior,opp_npxga_prior,goals_scored,team_ppg_venue
0,2016-09-10,785,Alaves,Barcelona,0,13.0,0,0.468085,0.583575,1.647879,3.452851,0.368976,2,1.000000
1,2016-09-19,799,Alaves,Deportivo La Coruna,1,8.0,0,1.045208,0.483288,1.522256,0.783360,0.637607,0,1.000000
2,2016-09-26,819,Alaves,Granada,1,5.0,0,0.800011,0.431351,1.047886,1.417881,1.777827,3,1.000000
3,2016-10-01,822,Alaves,Sevilla,0,4.0,0,1.292917,0.661661,0.774723,1.404738,0.854369,1,1.321837
4,2016-10-16,835,Alaves,Malaga,1,14.0,0,1.233474,0.731754,1.356628,1.161948,0.911206,1,1.689159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8856,2026-04-23,4499,Villarreal,Real Oviedo,0,11.0,1,1.627528,1.729207,0.963495,1.211083,1.358184,1,1.503412
8857,2026-04-26,4508,Villarreal,Celta Vigo,1,4.0,0,1.541627,1.773236,1.404591,1.390064,1.351968,2,2.392719
8858,2026-05-02,4511,Villarreal,Levante,1,5.0,1,1.599282,1.820768,1.468569,1.096784,0.922344,5,2.412759
8859,2026-05-10,4525,Villarreal,Mallorca,0,9.0,0,2.020615,1.877054,1.305188,1.355270,1.614404,1,1.547142


In [19]:
model_df[
    (model_df["team"] == "Burnley") &
    (model_df["match_date"].dt.year == 2026)
]

,match_date,fixture_id,team,opponent,is_home,opp_rest_days,opp_is_promoted,team_att_prior,team_xgf_prior,team_npxga_prior,opp_att_prior,opp_npxga_prior,goals_scored,team_ppg_venue


# **Output**

In [20]:
# ===============================
# MULTI-SHEET EXCEL OUTPUT (SELF-CONTAINED; INCLUDES FIXTURE INPUT)
# ===============================

from datetime import datetime
out_path = os.path.join(OUTPUTS, "Liga", f"liga_predictions_{datetime.today().strftime('%d_%m')}.xlsx")

import numpy as np
import pandas as pd
import math
from io import StringIO
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ---------- Guardrails ----------
if "final_model" not in globals():
    raise NameError("final_model not found. Run your final model training block first.")
if "best_priors_params" not in globals():
    raise NameError("best_priors_params not found. Run grid search best selection block first.")
if "FEATURE_COLS" not in globals():
    raise NameError("FEATURE_COLS not found. Ensure FEATURE_COLS matches training feature set used in grid search.")
if "team_matches" not in globals():
    raise NameError("team_matches not found.")
if "compute_priors" not in globals():
    raise NameError("compute_priors not found.")
if "all_comp" not in globals():
    raise NameError("all_comp not found. Run the rest days block first.")
if "promoted_current_season" not in globals():
    raise NameError("promoted_current_season not found. Run the promoted teams block first.")

# ---------- Fixture input ----------
FIXTURES_CSV_PATH = os.path.join(INPUTS, "Fixtures", "la_liga_fixtures.csv")
fixtures_upcoming = pd.read_csv(FIXTURES_CSV_PATH)
fixtures_upcoming["Date"] = pd.to_datetime(fixtures_upcoming["Date"])

# ---------- Production model — retrain on ALL data ----------
_priors_full   = compute_priors(team_matches, priors_params=best_priors_params)
_model_df_full = build_model_table(_priors_full, best_priors_params)
NON_FEATURE_COLS = {"match_date", "fixture_id", "team", "opponent", "goals_scored"}

X_all = _model_df_full[[c for c in _model_df_full.columns if c not in NON_FEATURE_COLS]]
y_all = _model_df_full["goals_scored"].astype(int)

production_model = XGBRegressor(**XGB_PARAMS)
production_model.fit(X_all, y_all)
print(f"Production model trained on {len(X_all)} rows (all available data)")
print(f"Features used: {list(X_all.columns)}")

# ---------- Time split config ----------
TRAIN_END_DATE = "2022-06-30"
VAL_END_DATE   = "2024-06-30"

def time_split(df):
    df = df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])
    train = df[df["match_date"] <= TRAIN_END_DATE]
    val   = df[(df["match_date"] > TRAIN_END_DATE) & (df["match_date"] <= VAL_END_DATE)]
    test  = df[df["match_date"] > VAL_END_DATE]
    return train, val, test

# ---------- Rest days helper ----------
def get_rest_days(team_name, fixture_date):
    """Days since team's last match (any competition) before fixture_date."""
    if len(all_comp) == 0:
        return np.nan
    past = all_comp[
        (all_comp["team"] == team_name) &
        (all_comp["date"] < pd.Timestamp(fixture_date).normalize())
    ]
    if len(past) == 0:
        return np.nan
    return int((pd.Timestamp(fixture_date).normalize() - past["date"].max()).days)

# ---------- Poisson / helpers ----------
GOAL_BUCKET_MAX = 6

def poisson_probs(lam, max_goal=GOAL_BUCKET_MAX):
    lam = max(float(lam), 1e-12)
    probs = [math.exp(-lam) * (lam**k) / math.factorial(k) for k in range(max_goal)]
    probs.append(max(1.0 - sum(probs), 0.0))
    return np.array(probs, dtype=float)

def prob_ge_from_bucket_probs(probs, threshold):
    need = int(math.floor(threshold) + 1)
    if need <= 0: return 1.0
    if need >= 6: return float(probs[-1])
    return float(np.sum(probs[need:6]) + probs[-1])

def safe_sheet_name(s):
    for ch in ['[',']',':','*','?','/','\\']:
        s = s.replace(ch, '')
    return s.strip()[:31]

def safe_odds(p):
    p = float(p)
    return np.nan if p <= 0 else 1.0 / p

# ---------- League averages ----------
def team_matches_to_match_level(df_team_matches):
    need_cols = {"fixture_id", "match_date", "season", "team", "is_home", "goals_for"}
    missing = need_cols - set(df_team_matches.columns)
    if missing:
        raise ValueError(f"team_matches missing: {sorted(missing)}")
    df = df_team_matches.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])
    df["is_home"] = df["is_home"].astype(int)
    def build_row(g):
        hr = g[g["is_home"]==1]; ar = g[g["is_home"]==0]
        return pd.Series({
            "fixture_id": g.name,
            "match_date": g["match_date"].iloc[0],
            "season":     g["season"].iloc[0],
            "home_team":  hr["team"].iloc[0] if len(hr) else np.nan,
            "away_team":  ar["team"].iloc[0] if len(ar) else np.nan,
            "home_goals": float(hr["goals_for"].iloc[0]) if len(hr) else np.nan,
            "away_goals": float(ar["goals_for"].iloc[0]) if len(ar) else np.nan,
        })
    match_df = df.groupby("fixture_id", sort=False).apply(build_row).reset_index(drop=True)
    return match_df.dropna(subset=["home_goals","away_goals","match_date"]).reset_index(drop=True)

def compute_val_league_averages_from_team_matches(team_matches_df):
    match_df = team_matches_to_match_level(team_matches_df)
    _, val_df, _ = time_split(match_df)
    hg = val_df["home_goals"].astype(float)
    ag = val_df["away_goals"].astype(float)
    total = hg + ag
    def rate(cond): return float(np.mean(cond)) if len(val_df) else np.nan
    return {
        "BTTS":           rate((hg > 0) & (ag > 0)),
        "Over 0.5":       rate(total >= 1),
        "Over 1.5":       rate(total >= 2),
        "Over 2.5":       rate(total >= 3),
        "Over 3.5":       rate(total >= 4),
        "Home Win":       rate(hg > ag),
        "Draw":           rate(hg == ag),
        "Away Win":       rate(hg < ag),
        "Home Over 0.5":  rate(hg >= 1),
        "Home Over 1.5":  rate(hg >= 2),
        "Home Over 2.5":  rate(hg >= 3),
        "Home Over 3.5":  rate(hg >= 4),
        "Away Over 0.5":  rate(ag >= 1),
        "Away Over 1.5":  rate(ag >= 2),
        "Away Over 2.5":  rate(ag >= 3),
        "Away Over 3.5":  rate(ag >= 4),
    }

league_avgs_val = compute_val_league_averages_from_team_matches(team_matches)

# ---------- Feature display map ----------
FEATURE_DISPLAY_MAP = {
    "team_att_prior":   ("Goals For (gf)",  "gf_prior"),
    "team_xgf_prior":   ("xG For",          "xgf_prior"),
    "team_npxga_prior": ("npxG Against",    "npxga_prior"),
    "team_ppg_venue":   None,
}
active_team_features = [c for c in FEATURE_COLS if c in FEATURE_DISPLAY_MAP]

# ---------- Priors helpers ----------
def build_team_priors_lookup(priors_df):
    df = priors_df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])
    df = df.sort_values(["team","match_date","fixture_id"]).reset_index(drop=True)
    for src, dst in [
        ("gf_prior",       "gf_prior"),
        ("xgf_prior",      "xgf_prior"),
        ("npxga_prior",    "npxga_prior"),
        ("ppg_home_prior", "ppg_home_prior"),
        ("ppg_away_prior", "ppg_away_prior"),
    ]:
        df[dst] = df[src].astype(float)
    return df[["fixture_id","match_date","team",
               "gf_prior","xgf_prior","npxga_prior",
               "ppg_home_prior","ppg_away_prior"]]

def latest_team_priors_before_date(team_priors_df, team_name, as_of_date):
    sub = team_priors_df[
        (team_priors_df["team"] == team_name) &
        (team_priors_df["match_date"] < as_of_date)
    ]
    keys = ["gf_prior","xgf_prior","npxga_prior","ppg_home_prior","ppg_away_prior"]
    if len(sub) == 0:
        return {k: np.nan for k in keys}
    row = sub.iloc[-1]
    return {k: float(row[k]) for k in keys}

def predict_lambdas_from_priors(home_priors, away_priors, home_rest, away_rest,
                                 home_is_promoted, away_is_promoted, model, feature_cols):
    all_home = {
        "is_home":           1,
        "opp_rest_days":     away_rest,
        "opp_is_promoted":   away_is_promoted,  # from home team's perspective, opp is away team
        "team_att_prior":    home_priors["gf_prior"],
        "team_xgf_prior":    home_priors["xgf_prior"],
        "team_npxga_prior":  home_priors["npxga_prior"],
        "team_ppg_venue":    home_priors["ppg_home_prior"],
        "opp_att_prior":     away_priors["gf_prior"],
        "opp_npxga_prior":   away_priors["npxga_prior"],
    }
    all_away = {
        "is_home":           0,
        "opp_rest_days":     home_rest,
        "opp_is_promoted":   home_is_promoted,  # from away team's perspective, opp is home team
        "team_att_prior":    away_priors["gf_prior"],
        "team_xgf_prior":    away_priors["xgf_prior"],
        "team_npxga_prior":  away_priors["npxga_prior"],
        "team_ppg_venue":    away_priors["ppg_away_prior"],
        "opp_att_prior":     home_priors["gf_prior"],
        "opp_npxga_prior":   home_priors["npxga_prior"],
    }
    X_home = pd.DataFrame([[all_home.get(c, np.nan) for c in feature_cols]], columns=feature_cols)
    X_away = pd.DataFrame([[all_away.get(c, np.nan) for c in feature_cols]], columns=feature_cols)
    return max(float(model.predict(X_home)[0]), 1e-9), max(float(model.predict(X_away)[0]), 1e-9)

# ---------- Styling ----------
thin        = Side(style="thin", color="3A3A3A")
border_thin = Border(left=thin, right=thin, top=thin, bottom=thin)

fill_header = PatternFill("solid", fgColor="D9ECFF")
fill_card   = PatternFill("solid", fgColor="F2F6FF")
fill_tableh = PatternFill("solid", fgColor="EAF5EA")
fill_max    = PatternFill("solid", fgColor="FFF59D")
fill_title  = PatternFill("solid", fgColor="1F4E79")
fill_g3     = PatternFill("solid", fgColor="1B5E20")
fill_g2     = PatternFill("solid", fgColor="2E7D32")
fill_g1     = PatternFill("solid", fgColor="C8E6C9")
fill_n0     = PatternFill("solid", fgColor="FFFFFF")
fill_r1     = PatternFill("solid", fgColor="FFCDD2")
fill_r2     = PatternFill("solid", fgColor="E57373")
fill_r3     = PatternFill("solid", fgColor="B71C1C")

font_title = Font(color="FFFFFF", bold=True, size=14)
font_h     = Font(bold=True)
font_b     = Font(bold=True)

align_c = Alignment(horizontal="center", vertical="center", wrap_text=True)
align_l = Alignment(horizontal="left",   vertical="center", wrap_text=True)

def set_block(ws, r1, c1, r2, c2, fill=None, border=True):
    for r in range(r1, r2+1):
        for c in range(c1, c2+1):
            cell = ws.cell(r, c)
            if fill:   cell.fill = fill
            if border: cell.border = border_thin
            cell.alignment = align_c

def write_kv(ws, start_row, start_col, items):
    r = start_row
    for k, v, fmt in items:
        c1 = ws.cell(r, start_col);  c2 = ws.cell(r, start_col+1)
        c1.value = k;                 c1.font = font_b
        c1.alignment = align_l;      c1.border = border_thin
        c2.value = v;                 c2.alignment = align_c
        c2.border = border_thin
        if fmt: c2.number_format = fmt
        r += 1

def fill_by_delta(delta):
    if np.isnan(delta): return fill_n0
    if delta >=  0.15:  return fill_g3
    if delta >=  0.08:  return fill_g2
    if delta >=  0.03:  return fill_g1
    if delta <= -0.15:  return fill_r3
    if delta <= -0.08:  return fill_r2
    if delta <= -0.03:  return fill_r1
    return fill_n0

def style_prob_cell_vs_avg(cell, prob, avg):
    cell.value         = float(prob)
    cell.number_format = "0.00%"
    cell.border        = border_thin
    cell.alignment     = align_c
    cell.fill          = fill_by_delta(float(prob) - float(avg)) if avg == avg else fill_n0
    return cell

def write_header_cell(ws, cell_ref, value, span_end=None):
    ws[cell_ref].value     = value
    ws[cell_ref].fill      = fill_header
    ws[cell_ref].font      = font_h
    ws[cell_ref].alignment = align_c
    ws[cell_ref].border    = border_thin
    if span_end:
        ws.merge_cells(f"{cell_ref}:{span_end}")

# ---------- Build priors once ----------
priors_df      = compute_priors(team_matches, priors_params=best_priors_params)
team_priors_df = build_team_priors_lookup(priors_df)

labels = ["0","1","2","3","4","5","6+"]

league_items = [
    ("BTTS",          "BTTS"),
    ("Over 0.5",      "Over 0.5"),
    ("Over 1.5",      "Over 1.5"),
    ("Over 2.5",      "Over 2.5"),
    ("Over 3.5",      "Over 3.5"),
    ("Home Win",      "Home Win"),
    ("Draw",          "Draw"),
    ("Away Win",      "Away Win"),
    ("Home Over 0.5", "Home Over 0.5"),
    ("Home Over 1.5", "Home Over 1.5"),
    ("Home Over 2.5", "Home Over 2.5"),
    ("Home Over 3.5", "Home Over 3.5"),
    ("Away Over 0.5", "Away Over 0.5"),
    ("Away Over 1.5", "Away Over 1.5"),
    ("Away Over 2.5", "Away Over 2.5"),
    ("Away Over 3.5", "Away Over 3.5"),
]

wrote_any_sheet = False

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    for _, fx in fixtures_upcoming.iterrows():
        dt       = pd.to_datetime(fx["Date"])
        time_utc = fx["Start Time (UTC)"]
        home     = fx["Home Team"]
        away     = fx["Away Team"]

        home_priors      = latest_team_priors_before_date(team_priors_df, home, dt)
        away_priors      = latest_team_priors_before_date(team_priors_df, away, dt)
        home_rest        = get_rest_days(home, dt)
        away_rest        = get_rest_days(away, dt)
        home_is_promoted = 1 if home in promoted_current_season else 0
        away_is_promoted = 1 if away in promoted_current_season else 0

        lam_home, lam_away = predict_lambdas_from_priors(
            home_priors, away_priors, home_rest, away_rest,
            home_is_promoted, away_is_promoted, production_model, FEATURE_COLS
        )

        # ---------- Dynamic stat rows ----------
        def build_stat_rows(priors, is_home_side):
            rows = []
            for col in active_team_features:
                if col == "team_ppg_venue":
                    rows.append(("PPG Home", priors.get("ppg_home_prior", np.nan), "0.000"))
                    rows.append(("PPG Away", priors.get("ppg_away_prior", np.nan), "0.000"))
                else:
                    label, prior_key = FEATURE_DISPLAY_MAP[col]
                    rows.append((label, priors.get(prior_key, np.nan), "0.000"))
            return rows

        home_stat_rows = build_stat_rows(home_priors, is_home_side=True)
        home_stat_rows.append(("Rest Days",          home_rest,        "0"))
        home_stat_rows.append(("Promoted",           home_is_promoted, "0"))
        home_stat_rows.append(("Expected Goals (λ)", lam_home,         "0.000"))

        away_stat_rows = build_stat_rows(away_priors, is_home_side=False)
        away_stat_rows.append(("Rest Days",          away_rest,        "0"))
        away_stat_rows.append(("Promoted",           away_is_promoted, "0"))
        away_stat_rows.append(("Expected Goals (λ)", lam_away,         "0.000"))

        n_stat_rows = len(home_stat_rows)

        # ---------- Row anchors ----------
        card_bottom       = 2 + n_stat_rows
        matrix_title_row  = card_bottom + 2
        matrix_header_row = matrix_title_row + 1
        matrix_start_row  = matrix_header_row + 1
        matrix_end_row    = matrix_start_row + len(labels) - 1

        btts_1x2_row    = matrix_start_row + 4 + 2
        odds_title_row  = matrix_end_row + 2
        odds_header_row = odds_title_row + 1
        odds_start_row  = odds_header_row + 1
        odds_btts_row   = odds_start_row + len(labels) + 2

        league_header_row = matrix_title_row
        league_data_start = matrix_header_row

        # ---------- Build sheet ----------
        ph = poisson_probs(lam_home)
        pa = poisson_probs(lam_away)
        score_matrix = np.outer(ph, pa)

        max_idx = np.unravel_index(np.argmax(score_matrix), score_matrix.shape)
        ml_home = labels[max_idx[0]]; ml_away = labels[max_idx[1]]
        ml_prob = float(score_matrix[max_idx])

        btts_yes = float(np.sum(score_matrix[1:, 1:]))
        btts_no  = 1.0 - btts_yes

        idx_vals    = np.array([0,1,2,3,4,5,6])
        totals      = idx_vals[:,None] + idx_vals[None,:]
        match_overs = {t: float(np.sum(score_matrix[totals >= int(math.floor(t)+1)])) for t in [0.5,1.5,2.5,3.5]}
        home_over   = {t: prob_ge_from_bucket_probs(ph, t) for t in [0.5,1.5,2.5,3.5]}
        away_over   = {t: prob_ge_from_bucket_probs(pa, t) for t in [0.5,1.5,2.5,3.5]}
        home_win    = float(np.sum(np.tril(score_matrix, k=-1)))
        draw        = float(np.sum(np.diag(score_matrix)))
        away_win    = float(np.sum(np.triu(score_matrix, k=1)))

        sheet = safe_sheet_name(f"{dt.strftime('%d %b')} {home} v {away}")
        pd.DataFrame({"": []}).to_excel(writer, sheet_name=sheet, index=False)
        ws = writer.book[sheet]
        wrote_any_sheet = True

        ws.freeze_panes = ws.cell(matrix_header_row, 1)

        # ── Title bar ──────────────────────────────────────────────────────────
        ws.merge_cells("A1:O1")
        ws["A1"].value     = f"{home} vs {away} • {dt.date().isoformat()} {time_utc} UTC"
        ws["A1"].fill      = fill_title
        ws["A1"].font      = font_title
        ws["A1"].alignment = align_c

        # ── Fixture block (cols A:C) ───────────────────────────────────────────
        set_block(ws, 2, 1, card_bottom, 3, fill=fill_card)
        write_kv(ws, 2, 1, [
            ("Start Time (UTC)", time_utc, None),
            ("Home Team",        home,     None),
            ("Away Team",        away,     None),
        ])

        # ── Home card (cols D:E) ───────────────────────────────────────────────
        ws.merge_cells("D2:E2")
        ws["D2"].value     = home
        ws["D2"].fill      = fill_header
        ws["D2"].font      = font_h
        ws["D2"].alignment = align_c
        set_block(ws, 2, 4, card_bottom, 5, fill=fill_card)
        write_kv(ws, 3, 4, home_stat_rows)

        # ── vs ─────────────────────────────────────────────────────────────────
        mid_row = 2 + n_stat_rows // 2
        ws.cell(mid_row, 6).value     = "vs"
        ws.cell(mid_row, 6).font      = Font(bold=True, size=12)
        ws.cell(mid_row, 6).alignment = align_c

        # ── Away card (cols G:H) ───────────────────────────────────────────────
        ws.merge_cells("G2:H2")
        ws["G2"].value     = away
        ws["G2"].fill      = fill_header
        ws["G2"].font      = font_h
        ws["G2"].alignment = align_c
        set_block(ws, 2, 7, card_bottom, 8, fill=fill_card)
        write_kv(ws, 3, 7, away_stat_rows)

        # ── Most likely score (cols L:N) ───────────────────────────────────────
        ws.merge_cells("L2:N2")
        ws["L2"].value     = "Most Likely Score"
        ws["L2"].fill      = fill_header
        ws["L2"].font      = font_h
        ws["L2"].alignment = align_c
        set_block(ws, 2, 12, card_bottom, 14, fill=fill_card)

        mls_data_row = 3
        for label, val, fmt in [
            (home,   ml_home, None),
            (away,   ml_away, None),
            ("Prob", ml_prob, "0.00%"),
        ]:
            ws.cell(mls_data_row, 12).value     = label
            ws.cell(mls_data_row, 12).font      = font_b
            ws.cell(mls_data_row, 12).alignment = align_c
            ws.cell(mls_data_row, 12).border    = border_thin
            ws.cell(mls_data_row, 13).value     = val
            ws.cell(mls_data_row, 13).alignment = align_c
            ws.cell(mls_data_row, 13).border    = border_thin
            if fmt: ws.cell(mls_data_row, 13).number_format = fmt
            mls_data_row += 1

        # ── League averages panel (cols P:Q) ───────────────────────────────────
        ws.merge_cells(f"P{league_header_row}:Q{league_header_row}")
        ws[f"P{league_header_row}"].value     = "League Averages (VAL)"
        ws[f"P{league_header_row}"].fill      = fill_header
        ws[f"P{league_header_row}"].font      = font_h
        ws[f"P{league_header_row}"].alignment = align_c
        ws[f"P{league_header_row}"].border    = border_thin

        for i, (label, key) in enumerate(league_items):
            rr = league_data_start + i
            c1 = ws.cell(rr, 16); c2 = ws.cell(rr, 17)
            c1.value = label;      c1.font = font_b
            c1.alignment = align_l; c1.border = border_thin
            v = league_avgs_val.get(key, np.nan)
            c2.value         = float(v) if v == v else np.nan
            c2.number_format = "0.00%"
            c2.alignment     = align_c
            c2.border        = border_thin

        ws.column_dimensions["P"].width = 20
        ws.column_dimensions["Q"].width = 12

        # ── Score matrix ───────────────────────────────────────────────────────
        ws.merge_cells(f"A{matrix_title_row}:H{matrix_title_row}")
        ws[f"A{matrix_title_row}"].value     = "Scoreline Probability Matrix"
        ws[f"A{matrix_title_row}"].fill      = fill_header
        ws[f"A{matrix_title_row}"].font      = font_h
        ws[f"A{matrix_title_row}"].alignment = align_c
        ws[f"A{matrix_title_row}"].border    = border_thin

        ws.cell(matrix_header_row, 1).value     = ""
        ws.cell(matrix_header_row, 1).fill      = fill_tableh
        ws.cell(matrix_header_row, 1).border    = border_thin
        ws.cell(matrix_header_row, 1).alignment = align_c

        for j, lab in enumerate(labels):
            cell = ws.cell(matrix_header_row, 2+j)
            cell.value = f"{away} {lab}"; cell.fill = fill_tableh
            cell.font = font_b; cell.alignment = align_c; cell.border = border_thin

        for i, lab in enumerate(labels):
            r = matrix_start_row + i
            ws.cell(r, 1).value     = f"{home} {lab}"
            ws.cell(r, 1).fill      = fill_tableh
            ws.cell(r, 1).font      = font_b
            ws.cell(r, 1).alignment = align_c
            ws.cell(r, 1).border    = border_thin
            for j in range(len(labels)):
                cell = ws.cell(r, 2+j)
                cell.value         = float(score_matrix[i, j])
                cell.number_format = "0.00%"
                cell.alignment     = align_c
                cell.border        = border_thin

        ws.cell(matrix_start_row + max_idx[0], 2 + max_idx[1]).fill = fill_max
        ws.cell(matrix_start_row + max_idx[0], 2 + max_idx[1]).font = Font(bold=True)

        # ── Over tables ────────────────────────────────────────────────────────
        over_map = {
            "Over Match":      (10, 11, {0.5:"Over 0.5", 1.5:"Over 1.5", 2.5:"Over 2.5", 3.5:"Over 3.5"}, match_overs),
            f"Over {home}":    (12, 13, {0.5:"Home Over 0.5", 1.5:"Home Over 1.5", 2.5:"Home Over 2.5", 3.5:"Home Over 3.5"}, home_over),
            f"Over {away}":    (14, 15, {0.5:"Away Over 0.5", 1.5:"Away Over 1.5", 2.5:"Away Over 2.5", 3.5:"Away Over 3.5"}, away_over),
        }
        for title, (col_l, col_v, avg_keys, overs_dict) in over_map.items():
            set_block(ws, matrix_header_row, col_l, matrix_header_row+5, col_v, fill=fill_card)
            hdr = ws.cell(matrix_header_row, col_l)
            hdr.value = title; hdr.fill = fill_header; hdr.font = font_h
            hdr.alignment = align_c; hdr.border = border_thin
            ws.merge_cells(start_row=matrix_header_row, start_column=col_l,
                           end_row=matrix_header_row, end_column=col_v)
            for idx, t in enumerate([0.5,1.5,2.5,3.5]):
                rr = matrix_start_row + idx
                lbl_cell = ws.cell(rr, col_l)
                lbl_cell.value = t; lbl_cell.border = border_thin; lbl_cell.alignment = align_c
                style_prob_cell_vs_avg(ws.cell(rr, col_v), overs_dict[t], league_avgs_val.get(avg_keys[t], np.nan))

        # ── BTTS ───────────────────────────────────────────────────────────────
        btts_row = matrix_start_row + 5
        set_block(ws, btts_row, 10, btts_row+2, 11, fill=fill_card)
        hdr = ws.cell(btts_row, 10)
        hdr.value = "BTTS"; hdr.fill = fill_header; hdr.font = font_h
        hdr.alignment = align_c; hdr.border = border_thin
        ws.merge_cells(start_row=btts_row, start_column=10, end_row=btts_row, end_column=11)
        for i, (label, prob, avg) in enumerate([
            ("Yes", btts_yes, league_avgs_val.get("BTTS", np.nan)),
            ("No",  btts_no,  1.0 - float(league_avgs_val["BTTS"]) if league_avgs_val.get("BTTS")==league_avgs_val.get("BTTS") else np.nan),
        ]):
            r = btts_row + 1 + i
            ws.cell(r, 10).value = label; ws.cell(r, 10).font = font_b
            ws.cell(r, 10).border = border_thin; ws.cell(r, 10).alignment = align_c
            style_prob_cell_vs_avg(ws.cell(r, 11), prob, avg)

        # ── 1X2 ────────────────────────────────────────────────────────────────
        set_block(ws, btts_row, 14, btts_row+3, 15, fill=fill_card)
        hdr = ws.cell(btts_row, 14)
        hdr.value = "1X2"; hdr.fill = fill_header; hdr.font = font_h
        hdr.alignment = align_c; hdr.border = border_thin
        ws.merge_cells(start_row=btts_row, start_column=14, end_row=btts_row, end_column=15)
        for i, (label, prob, key) in enumerate([
            ("Home", home_win, "Home Win"),
            ("Draw", draw,     "Draw"),
            ("Away", away_win, "Away Win"),
        ]):
            r = btts_row + 1 + i
            ws.cell(r, 14).value = label; ws.cell(r, 14).font = font_b
            ws.cell(r, 14).border = border_thin; ws.cell(r, 14).alignment = align_c
            style_prob_cell_vs_avg(ws.cell(r, 15), prob, league_avgs_val.get(key, np.nan))

        # ── Odds block ─────────────────────────────────────────────────────────
        ws.merge_cells(f"A{odds_title_row}:H{odds_title_row}")
        ws[f"A{odds_title_row}"].value     = "Expected Odds (1 / Probability)"
        ws[f"A{odds_title_row}"].fill      = fill_header
        ws[f"A{odds_title_row}"].font      = font_h
        ws[f"A{odds_title_row}"].alignment = align_c
        ws[f"A{odds_title_row}"].border    = border_thin

        ws.cell(odds_header_row, 1).value     = ""
        ws.cell(odds_header_row, 1).fill      = fill_tableh
        ws.cell(odds_header_row, 1).border    = border_thin
        ws.cell(odds_header_row, 1).alignment = align_c

        for j, lab in enumerate(labels):
            cell = ws.cell(odds_header_row, 2+j)
            cell.value = f"{away} {lab}"; cell.fill = fill_tableh
            cell.font = font_b; cell.alignment = align_c; cell.border = border_thin

        for i, lab in enumerate(labels):
            r = odds_start_row + i
            ws.cell(r, 1).value     = f"{home} {lab}"
            ws.cell(r, 1).fill      = fill_tableh
            ws.cell(r, 1).font      = font_b
            ws.cell(r, 1).alignment = align_c
            ws.cell(r, 1).border    = border_thin
            for j in range(len(labels)):
                cell = ws.cell(r, 2+j)
                cell.value         = safe_odds(float(score_matrix[i, j]))
                cell.number_format = "0.00"
                cell.alignment     = align_c
                cell.border        = border_thin

        # Odds over tables
        odds_over_map = {
            "Over Match (Odds)":   (10, 11, match_overs),
            f"Over {home} (Odds)": (12, 13, home_over),
            f"Over {away} (Odds)": (14, 15, away_over),
        }
        for title, (col_l, col_v, overs_dict) in odds_over_map.items():
            set_block(ws, odds_header_row, col_l, odds_header_row+5, col_v, fill=fill_card)
            hdr = ws.cell(odds_header_row, col_l)
            hdr.value = title; hdr.fill = fill_header; hdr.font = font_h
            hdr.alignment = align_c; hdr.border = border_thin
            ws.merge_cells(start_row=odds_header_row, start_column=col_l,
                           end_row=odds_header_row, end_column=col_v)
            for idx, t in enumerate([0.5,1.5,2.5,3.5]):
                rr = odds_start_row + idx
                ws.cell(rr, col_l).value     = t
                ws.cell(rr, col_l).border    = border_thin
                ws.cell(rr, col_l).alignment = align_c
                ws.cell(rr, col_v).value         = safe_odds(overs_dict[t])
                ws.cell(rr, col_v).number_format = "0.00"
                ws.cell(rr, col_v).border        = border_thin
                ws.cell(rr, col_v).alignment     = align_c

        # Odds BTTS
        set_block(ws, odds_btts_row, 10, odds_btts_row+2, 11, fill=fill_card)
        hdr = ws.cell(odds_btts_row, 10)
        hdr.value = "BTTS (Odds)"; hdr.fill = fill_header; hdr.font = font_h
        hdr.alignment = align_c; hdr.border = border_thin
        ws.merge_cells(start_row=odds_btts_row, start_column=10, end_row=odds_btts_row, end_column=11)
        for i, (label, prob) in enumerate([("Yes", btts_yes), ("No", btts_no)]):
            r = odds_btts_row + 1 + i
            ws.cell(r, 10).value = label; ws.cell(r, 10).font = font_b
            ws.cell(r, 10).border = border_thin; ws.cell(r, 10).alignment = align_c
            ws.cell(r, 11).value         = safe_odds(prob)
            ws.cell(r, 11).number_format = "0.00"
            ws.cell(r, 11).border        = border_thin
            ws.cell(r, 11).alignment     = align_c

        # Odds 1X2
        set_block(ws, odds_btts_row, 14, odds_btts_row+3, 15, fill=fill_card)
        hdr = ws.cell(odds_btts_row, 14)
        hdr.value = "1X2 (Odds)"; hdr.fill = fill_header; hdr.font = font_h
        hdr.alignment = align_c; hdr.border = border_thin
        ws.merge_cells(start_row=odds_btts_row, start_column=14, end_row=odds_btts_row, end_column=15)
        for i, (label, prob) in enumerate([("Home", home_win), ("Draw", draw), ("Away", away_win)]):
            r = odds_btts_row + 1 + i
            ws.cell(r, 14).value = label; ws.cell(r, 14).font = font_b
            ws.cell(r, 14).border = border_thin; ws.cell(r, 14).alignment = align_c
            ws.cell(r, 15).value         = safe_odds(prob)
            ws.cell(r, 15).number_format = "0.00"
            ws.cell(r, 15).border        = border_thin
            ws.cell(r, 15).alignment     = align_c

        # ── Column widths ──────────────────────────────────────────────────────
        for col in range(1, 18):
            ws.column_dimensions[get_column_letter(col)].width = 15
        ws.column_dimensions["A"].width = 20
        ws.column_dimensions["F"].width = 6
        ws.column_dimensions["P"].width = 22
        ws.column_dimensions["Q"].width = 12

if not wrote_any_sheet:
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        pd.DataFrame({"Message": ["No fixtures provided."]}).to_excel(writer, sheet_name="Info", index=False)

print("Saved:", out_path)

Production model trained on 8861 rows (all available data)
Features used: ['is_home', 'opp_rest_days', 'opp_is_promoted', 'team_att_prior', 'team_xgf_prior', 'team_npxga_prior', 'opp_att_prior', 'opp_npxga_prior', 'team_ppg_venue']
Saved: /Users/patrickknott/Desktop/Football_Prediction_Project/Outputs/Liga/liga_predictions_17_05.xlsx
